In [29]:
%load_ext autoreload
%autoreload 2

from src import fsrs_simulator as fsrs
from src.algorithm import (
    CEFR_LEVELS,
    EXERCISE_TYPE_MAP,
    _HUMAN_PRESETS,
    GreedyFSRSSelector,
    QueueGenerator,
    QueueSelector,
    RandomSelector,
    BeamSelector,
    GreedySigmoidSelector,
    GreedyHSTUSelector
)

import random

# Seconds per day; FSRS uses days internally
_DAY = 86_400
# Base Unix timestamp so generated events look realistic
_BASE_TS = 1_600_000_000

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Анализ отдельной сессии: кривые узнавания и ERG по шагам

In [2]:
queue_gen = QueueGenerator()

In [3]:
import copy
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def shifted_sigmoid(x, shift=0.7, k=5):
    tanh = np.tanh(k*(x - shift))
    th_shift = np.tanh(-k * shift)
    scale = np.tanh(k * 1 - k * shift) - th_shift

    return  (tanh - th_shift) / scale


def simulate_session_with_analysis(
    cefr_level: str = "C1",
    n_steps: int = 60,
    selector_type=GreedyFSRSSelector,
    selector_kwargs: dict = None,
    seed: int = 42,
    step_size_days: float = 1.0,
    num_words=10,
):
    """
    Запускает одну сессию и на каждом шаге собирает:
    - retrievability (кривая узнавания) для каждого слова
    - ERG для каждого слова (макс. по типам упражнений)
    - слово-победитель по ERG

    step_size_days: шаг времени между упражнениями в днях.
        FSRS-стабильность после первого успешного повторения ≈ 0.4 дня,
        поэтому шаг в минутах (1/1440) даёт практически нулевое забывание
        на протяжении всей сессии. Дефолт 1.0 день показывает реалистичные
        кривые угасания retrievability.

    Returns
    -------
    df_retriev : step × word → retrievability
    df_erg     : step × word → erg, is_winner, exercised
    df_steps   : сводка по шагам (показанное слово, победитель ERG, успех, recall_prob)
    """
    random.seed(seed)

    preset = _HUMAN_PRESETS.get(cefr_level.upper(), _HUMAN_PRESETS["B1"])
    human = fsrs.Human(human_id=0, **preset)

    queue = queue_gen.construct_queue(num_words, selector_type, **(selector_kwargs or {}))
    human.reset_session()

    # Инициализируем все слова очереди заранее
    word_cache: dict[str, fsrs.Word] = {}
    for w_text in list(queue.selector.words.keys()):
        word_cache[w_text] = fsrs.Word(word_id=len(word_cache), text=w_text, translation="")

    retriev_rows, erg_rows, step_rows = [], [], []
    current_ts = 0.0

    for step in range(n_steps):
        if queue.is_finished():
            break

        # 1. Кривая узнавания — retrievability каждого слова в текущий момент
        for w_text, fsrs_w in word_cache.items():
            r = human._compute_retrievability(fsrs_w, current_ts) if fsrs_w.seen else 0.0
            retriev_rows.append({"step": step, "word": w_text, "retrievability": r, "sigm": shifted_sigmoid(r)})

        # 2. ERG для каждого слова — берём максимум по всем типам упражнений
        erg_snapshot: dict[str, float] = {}
        best_ex_snapshot: dict[str, str] = {}
        for w_text, fsrs_w in word_cache.items():
            best_erg, best_ex = -float("inf"), None
            for ex_type in fsrs.ExerciseType:
                erg = human.estimate_erg(fsrs_w, ex_type, current_ts)
                if erg > best_erg:
                    best_erg, best_ex = erg, ex_type
            erg_snapshot[w_text] = best_erg
            best_ex_snapshot[w_text] = best_ex.code if best_ex else ""

        winner = max(erg_snapshot, key=erg_snapshot.get)

        # 3. Выполняем упражнение
        ex = queue.produce_next_excercise()
        fsrs_w = word_cache[ex.word]
        ex_type = EXERCISE_TYPE_MAP.get(ex.exercise_class, fsrs.ExerciseType.TRANSLATE_EN_RU)
        result = human.attempt(fsrs_w, ex_type, current_ts)
        is_correct = result["success"]
        recall_prob = result["recall_prob"]  # вероятность клика (используется как soft target в лоссе)
        queue.progress(ex, is_correct)

        for w_text, erg_val in erg_snapshot.items():
            erg_rows.append({
                "step": step,
                "word": w_text,
                "erg": erg_val,
                "best_exercise": best_ex_snapshot[w_text],
                "is_winner": w_text == winner,
                "exercised": w_text == ex.word,
            })

        step_rows.append({
            "step": step,
            "exercised_word": ex.word,
            "erg_winner": winner,
            "success": is_correct,
            "recall_prob": recall_prob,
            "winner_matches_exercised": winner == ex.word,
        })

        current_ts += step_size_days

    return pd.DataFrame(retriev_rows), pd.DataFrame(erg_rows), pd.DataFrame(step_rows)


In [4]:
df_retriev, df_erg, df_steps = simulate_session_with_analysis(n_steps=60, seed=42)

match_rate = df_steps["winner_matches_exercised"].mean()
print(f"Шагов: {len(df_steps)}  |  Слов: {df_retriev['word'].nunique()}")
print(f"ERG-победитель совпал со случайным выбором: {match_rate:.1%}")
df_steps.head(10)


Шагов: 60  |  Слов: 10
ERG-победитель совпал со случайным выбором: 51.7%


,step,exercised_word,erg_winner,success,recall_prob,winner_matches_exercised
0,0,doubly,doubly,False,0.024298,True
1,1,attentively,attentively,False,0.023885,True
2,2,pluck,pluck,False,0.023473,True
3,3,nominee,nominee,False,0.023062,True
4,4,material,material,False,0.022652,True
5,5,fearsome,fearsome,False,0.022242,True
6,6,dig,dig,False,0.022220,True
7,7,crossly,crossly,False,0.022198,True
8,8,isolated,isolated,False,0.022176,True
9,9,bath,bath,False,0.022154,True


In [5]:
# График 1: Тепловая карта кривых узнавания (retrievability по шагам)

# Сортируем слова по шагу первого появления (когда retrievability > 0)
first_seen_step = (
    df_retriev[df_retriev["retrievability"] > 0]
    .groupby("word")["step"].min()
    .sort_values()
)
word_order = first_seen_step.index.tolist()

pivot_r = (
    df_retriev.pivot(index="word", columns="step", values="retrievability")
    .fillna(0)
    .loc[word_order]
)

fig = go.Figure(go.Heatmap(
    z=pivot_r.values,
    x=pivot_r.columns.tolist(),
    y=word_order,
    colorscale="Blues",
    colorbar=dict(title="Retrievability"),
    zmin=0, zmax=1,
))

exercised_pts = df_erg[df_erg["exercised"]]
fig.add_trace(go.Scatter(
    x=exercised_pts["step"],
    y=exercised_pts["word"],
    mode="markers",
    marker=dict(symbol="circle-open", size=8, color="red", line=dict(width=1.5)),
    name="Показано в упражнении",
))

fig.update_layout(
    title="Кривые узнавания: retrievability каждого слова по шагам<br><sup>○ = момент показа слова</sup>",
    xaxis_title="День",
    yaxis_title="Слово",
    height=720,
    legend=dict(orientation="h", y=-0.1),
)
fig.show()


In [6]:
# График 2: Тепловая карта ERG + победитель на каждом шаге

pivot_erg = (
    df_erg.pivot(index="word", columns="step", values="erg")
    .fillna(0)
    .loc[word_order]
)

winners_df = df_erg[df_erg["is_winner"]]

fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=pivot_erg.values,
    x=pivot_erg.columns.tolist(),
    y=word_order,
    colorscale="RdYlGn",
    colorbar=dict(title="ERG"),
    name="ERG",
))

fig.add_trace(go.Scatter(
    x=winners_df["step"],
    y=winners_df["word"],
    mode="markers",
    marker=dict(symbol="star", size=10, color="gold", line=dict(color="black", width=0.8)),
    name="ERG-победитель (★)",
))

fig.add_trace(go.Scatter(
    x=exercised_pts["step"],
    y=exercised_pts["word"],
    mode="markers",
    marker=dict(symbol="circle-open", size=9, color="white", line=dict(width=1.5)),
    name="Показано случайно (○)",
))

fig.update_layout(
    title="ERG по дням<br><sup>★ = слово-победитель по ERG  |  ○ = фактически показанное слово</sup>",
    xaxis_title="День",
    yaxis_title="Слово",
    height=720,
    legend=dict(orientation="h", y=-0.1),
)
fig.show()


In [7]:
# График 3: Таймлайн — ERG-победитель vs показанное слово + скользящая доля совпадений

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        "ERG-победитель (★) vs случайно показанное слово (○) по дням",
        "Скользящая доля совпадений ERG-победителя со случайным выбором",
    ],
    row_heights=[0.68, 0.32],
    vertical_spacing=0.14,
)

fig.add_trace(go.Scatter(
    x=df_steps["step"],
    y=df_steps["erg_winner"],
    mode="markers+lines",
    marker=dict(symbol="star", size=9, color="gold"),
    line=dict(color="rgba(255,200,0,0.25)", width=1),
    name="ERG-победитель",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_steps["step"],
    y=df_steps["exercised_word"],
    mode="markers",
    marker=dict(symbol="circle-open", size=7, color="steelblue"),
    name="Показано случайно",
), row=1, col=1)

window = 10
rolling_match = df_steps["winner_matches_exercised"].rolling(window, min_periods=1).mean()
fig.add_trace(go.Scatter(
    x=df_steps["step"],
    y=rolling_match,
    mode="lines",
    fill="tozeroy",
    fillcolor="rgba(0,180,80,0.15)",
    line=dict(color="green", width=2),
    name=f"Скользящее среднее (окно={window})",
), row=2, col=1)

fig.add_hline(y=rolling_match.mean(), line_dash="dash", line_color="gray",
              annotation_text=f"среднее {rolling_match.mean():.1%}", row=2, col=1)

fig.update_yaxes(title_text="Слово", row=1, col=1)
fig.update_yaxes(title_text="Доля совпадений", range=[0, 1], row=2, col=1)
fig.update_xaxes(title_text="День", row=2, col=1)
fig.update_layout(height=700, showlegend=True, legend=dict(orientation="h", y=-0.08))
fig.show()


In [8]:
# График 4: Кривые узнавания топ-N слов по максимальному ERG

TOP_N = 12
top_words = (
    df_erg.groupby("word")["erg"].max()
    .nlargest(TOP_N)
    .index.tolist()
)

df_top_r = df_retriev[df_retriev["word"].isin(top_words)]
df_top_ex = df_erg[(df_erg["exercised"]) & (df_erg["word"].isin(top_words))]

ret_lookup = df_top_r.set_index(["word", "step"])["retrievability"]

colors = px.colors.qualitative.Plotly

fig = go.Figure()
for i, word in enumerate(top_words):
    color = colors[i % len(colors)]
    sub = df_top_r[df_top_r["word"] == word].sort_values("step")

    fig.add_trace(go.Scatter(
        x=sub["step"], y=sub["retrievability"],
        mode="lines",
        name=word,
        line=dict(color=color, width=2),
        legendgroup=word,
    ))

    ex_steps = df_top_ex[df_top_ex["word"] == word]["step"].values
    ex_y = [ret_lookup.get((word, s), 0.0) for s in ex_steps]
    fig.add_trace(go.Scatter(
        x=ex_steps, y=ex_y,
        mode="markers",
        marker=dict(symbol="circle", size=9, color=color, line=dict(color="black", width=1)),
        showlegend=False,
        legendgroup=word,
        hovertemplate=f"{word}<br>день: %{{x}}<br>R: %{{y:.3f}}<extra></extra>",
    ))

fig.update_layout(
    title=f"Кривые узнавания топ-{TOP_N} слов по ERG<br><sup>● = момент показа слова в упражнении</sup>",
    xaxis_title="День",
    yaxis_title="Retrievability",
    yaxis=dict(range=[0, 1]),
    height=520,
    legend=dict(orientation="h", y=-0.28),
)
fig.show()


## Сравнение селекторов: средняя retrievability по шагам

In [41]:
from src.models import load_model, HSTUPredictor

predictor = HSTUPredictor(load_model("model.pt"))
predictor

In [52]:
SELECTOR_CONFIGS = [
    ("Random",        RandomSelector,        {}),
    ("Queue",         QueueSelector,         {}),
    ("GreedyFSRS",    GreedyFSRSSelector,    {"english_level": "B2", "step_size_days": 1.0}),
    ("GreedySigmoid", GreedySigmoidSelector, {"english_level": "B2", "step_size_days": 1.0}),
    ("GreedyHSTU",    GreedyHSTUSelector,    {"english_level": "B2", "step_size_days": 1.0, "predictor": predictor, "mode": "sigm"}),
]

N_SEEDS   = 5
N_STEPS   = 30
STEP_DAYS = 1.0
LEVEL     = "B2"
# Метрика оценки: "retrievability" — интерпретируемая [0,1] вероятность вспомнить слово.
# "sigm" — shifted-sigmoid трансформация той же величины (внутренний objective GreedySigmoid).
# Для сравнения селекторов всегда используем retrievability.
METRIC    = "sigm"

all_retriev: list[pd.DataFrame] = []

for name, sel_type, sel_kw in SELECTOR_CONFIGS:
    for seed in range(N_SEEDS):
        df_r, _, _ = simulate_session_with_analysis(
            cefr_level=LEVEL,
            n_steps=N_STEPS,
            selector_type=sel_type,
            selector_kwargs=sel_kw,
            seed=seed,
            step_size_days=STEP_DAYS,
            num_words=30,
        )
        df_r["selector"] = name
        df_r["seed"]     = seed
        all_retriev.append(df_r)

df_all = pd.concat(all_retriev, ignore_index=True)

mean_per_run = (
    df_all.groupby(["selector", "seed", "step"])[METRIC]
    .mean()
    .reset_index(name="mean_metric")
)

agg = (
    mean_per_run.groupby(["selector", "step"])["mean_metric"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
agg["ci95"] = 1.96 * agg["std"] / np.sqrt(agg["count"])

print(f"Метрика: {METRIC}  |  Финальный шаг {N_STEPS - 1}:")
print(agg[agg["step"] == N_STEPS - 1][["selector", "mean", "ci95"]].to_string(index=False))

Метрика: sigm  |  Финальный шаг 29:
     selector     mean     ci95
   GreedyFSRS 0.058865 0.001313
   GreedyHSTU 0.174154 0.043261
GreedySigmoid 0.158115 0.028127
        Queue 0.059251 0.002070
       Random 0.118194 0.027780


In [54]:
_COLORS = [
    ("#636EFA", "rgba(99,110,250,0.15)"),
    ("#EF553B", "rgba(239,85,59,0.15)"),
    ("#00CC96", "rgba(0,204,150,0.15)"),
    ("#AB63FA", "rgba(171,99,250,0.15)"),
    ("#FFA15A", "rgba(255,161,90,0.15)"),
    ("#19D3F3", "rgba(25,211,243,0.15)"),
]
PALETTE = {
    name: {"line": line, "fill": fill}
    for (name, _, _), (line, fill) in zip(SELECTOR_CONFIGS, _COLORS)
}

selectors_to_plot = [name for name, _, _ in SELECTOR_CONFIGS]

fig = go.Figure()

for name in selectors_to_plot:
    sub = agg[agg["selector"] == name].sort_values("step")
    line_color = PALETTE[name]["line"]
    fill_color = PALETTE[name]["fill"]

    # CI band
    fig.add_trace(go.Scatter(
        x=pd.concat([sub["step"], sub["step"].iloc[::-1]]),
        y=pd.concat([sub["mean"] + sub["ci95"], (sub["mean"] - sub["ci95"]).iloc[::-1]]),
        fill="toself",
        fillcolor=fill_color,
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
        hoverinfo="skip",
    ))

    fig.add_trace(go.Scatter(
        x=sub["step"],
        y=sub["mean"],
        mode="lines+markers",
        name=name,
        line=dict(color=line_color, width=2.5),
        marker=dict(size=5),
    ))

metric_label = "Retrievability" if METRIC == "retrievability" else "Sigmoid(Retrievability)"
fig.update_layout(
    title=(
        f"Средняя {metric_label} по всем словам сессии<br>"
        f"<sup>{N_STEPS} шагов · шаг = {STEP_DAYS} д · уровень {LEVEL} · "
        f"{N_SEEDS} прогонов, полоса = 95% CI</sup>"
    ),
    xaxis_title="Шаг",
    yaxis_title=f"Средняя {metric_label}",
    yaxis=dict(range=[0, 0.2]),
    height=480,
    legend=dict(title="Селектор"),
)
fig.show()